# **SESIÓN 3:** Autocorrelación, Calidad de Datos y Modelos Baseline
## Universidad Autónoma de Occidente
### Maestría en Inteligencia Artificial y Ciencias de Datos
**Instructor:** Dr. Ing. Sergio A. Cantillo

---

## Análisis de serie temporal semanal de dengue en formato Nixtla

Este notebook conserva la estructura pedagógica original de la sesión, pero adapta el flujo completo al dataset de dengue suministrado. El objetivo es construir una base técnica sólida para la **Fase 1 del proyecto de series temporales**: exploración, diagnóstico, autocorrelación, estacionariedad, modelos baseline y validación temporal.

### Objetivos analíticos

- Convertir el archivo CSV original de casos individuales a una serie semanal en formato **Nixtla**: `unique_id`, `ds`, `y`.
- Evaluar calidad de datos: continuidad temporal, semanas faltantes, nulos, duplicados conceptuales y valores atípicos.
- Caracterizar patrones de tendencia, estacionalidad, ciclos epidémicos y cambios de régimen.
- Diagnosticar estacionariedad mediante ADF, KPSS, transformaciones y análisis visual.
- Analizar dependencia temporal mediante ACF, PACF y Ljung-Box.
- Implementar modelos baseline reproducibles y comparables: `Naive`, `SeasonalNaive`, `WindowAverage` y `RandomWalkWithDrift`.
- Evaluar desempeño con métricas MAE, RMSE, MAPE y sMAPE, más diagnóstico de residuos.
- Complementar el split simple con validación cruzada temporal.

### Dataset

| Elemento | Descripción |
|---|---|
| Dominio | Vigilancia epidemiológica |
| Unidad original | Registros individuales de dengue |
| Frecuencia de modelado | Semanal |
| Variable objetivo | Número de casos semanales |
| Formato final | `unique_id`, `ds`, `y` |

> **Nota metodológica:** en epidemiología, los picos extremos no deben eliminarse automáticamente. En muchos casos representan brotes reales y, por tanto, contienen información crítica para vigilancia y pronóstico.


## ⚙️ PARTE 1: Configuración del Entorno

Esta sección prepara las librerías, configura el estilo visual y define funciones reutilizables. La idea es que el notebook sea reproducible, modular y fácil de mantener. Se usa un enfoque funcional: las transformaciones principales reciben dataframes como entrada y devuelven nuevos dataframes o tablas resumen sin modificar innecesariamente los objetos originales.


In [ ]:
# Instalación opcional para entornos tipo Colab.
# En un ambiente local ya configurado, esta celda puede omitirse.
!pip install statsforecast utilsforecast statsmodels scipy plotly -q


In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Iterable, Sequence
import warnings

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from scipy import stats
from scipy.stats import skew, kurtosis

from statsmodels.tsa.stattools import adfuller, kpss, acf, pacf
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

from statsforecast import StatsForecast
from statsforecast.models import Naive, SeasonalNaive, WindowAverage, RandomWalkWithDrift

import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

RANDOM_SEED = 42
SERIES_ID = "dengue_cali"
DATE_COLUMN = "fec_not"
TARGET_COLUMN = "y"
WEEKLY_FREQUENCY = "W-MON"
SEASON_LENGTH = 52
DEFAULT_TEST_SIZE_RATIO = 0.20

np.random.seed(RANDOM_SEED)


In [ ]:
# ============================================================
# Funciones generales de carga, preparación y diagnóstico
# ============================================================

def resolve_existing_path(candidate_paths: Sequence[str | Path]) -> Path:
    """Devuelve la primera ruta existente dentro de una lista de candidatos."""
    for candidate_path in candidate_paths:
        path = Path(candidate_path)
        if path.exists():
            return path
    raise FileNotFoundError(
        "No se encontró el archivo de datos. Ajusta DATA_PATH con la ubicación correcta del CSV."
    )


def load_raw_dengue_csv(csv_path: str | Path) -> pd.DataFrame:
    """Carga el CSV original de dengue sin transformar la granularidad."""
    raw_data = pd.read_csv(csv_path)
    return raw_data.copy()


def build_weekly_nixtla_series(
    raw_data: pd.DataFrame,
    date_column: str,
    series_id: str,
    week_anchor: str = "W-SUN",
) -> pd.DataFrame:
    """
    Convierte registros individuales a serie semanal en formato Nixtla.

    Parámetros
    ----------
    raw_data:
        DataFrame original, con al menos una columna de fecha.
    date_column:
        Nombre de la columna de fecha de notificación.
    series_id:
        Identificador único de la serie.
    week_anchor:
        Regla de pandas para construir semanas. `W-SUN` define semanas que terminan domingo;
        `start_time` devuelve el lunes, consistente con `W-MON`.
    """
    if date_column not in raw_data.columns:
        raise KeyError(f"La columna de fecha '{date_column}' no existe en el CSV.")

    valid_records = raw_data.copy()
    valid_records[date_column] = pd.to_datetime(valid_records[date_column], errors="coerce")
    valid_records = valid_records.dropna(subset=[date_column])

    valid_records["ds"] = valid_records[date_column].dt.to_period(week_anchor).dt.start_time

    weekly_series = (
        valid_records.groupby("ds")
        .size()
        .reset_index(name="y")
        .sort_values("ds")
        .reset_index(drop=True)
    )
    weekly_series.insert(0, "unique_id", series_id)
    return weekly_series[["unique_id", "ds", "y"]]


def complete_weekly_calendar(
    nixtla_df: pd.DataFrame,
    frequency: str = WEEKLY_FREQUENCY,
) -> pd.DataFrame:
    """Completa el calendario semanal y marca semanas faltantes."""
    complete_dates = pd.date_range(
        start=nixtla_df["ds"].min(),
        end=nixtla_df["ds"].max(),
        freq=frequency,
    )
    calendar = pd.DataFrame({"unique_id": nixtla_df["unique_id"].iloc[0], "ds": complete_dates})
    completed = calendar.merge(nixtla_df, on=["unique_id", "ds"], how="left")
    completed["is_missing_calendar"] = completed["y"].isna()
    return completed


def add_modeling_target_with_seasonal_imputation(
    completed_df: pd.DataFrame,
    target_column: str = "y",
    output_column: str = "y_model",
) -> pd.DataFrame:
    """
    Crea una columna auxiliar para análisis/modelado que requieren grilla regular.

    La serie observada `y` no se reemplaza. `y_model` se usa únicamente para procedimientos
    como descomposición, ACF/PACF, Ljung-Box y baselines.
    """
    result = completed_df.copy()
    result["week_of_year"] = result["ds"].dt.isocalendar().week.astype(int)
    seasonal_week_mean = result.groupby("week_of_year")[target_column].transform("mean")
    result[output_column] = (
        result[target_column]
        .fillna(seasonal_week_mean)
        .interpolate(method="linear")
        .bfill()
        .ffill()
    )
    return result


def summarize_nixtla_structure(nixtla_df: pd.DataFrame, raw_data: pd.DataFrame) -> pd.DataFrame:
    """Resume la estructura del dataset original y de la serie semanal final."""
    return pd.DataFrame({
        "Indicador": [
            "Filas originales del CSV",
            "Columnas originales del CSV",
            "Identificador de serie",
            "Frecuencia de trabajo",
            "Fecha inicial",
            "Fecha final",
            "Semanas observadas",
            "Total de casos agregados",
            "Columnas formato Nixtla",
        ],
        "Valor": [
            len(raw_data),
            ", ".join(raw_data.columns.astype(str)),
            nixtla_df["unique_id"].iloc[0],
            "Semanal",
            nixtla_df["ds"].min().date(),
            nixtla_df["ds"].max().date(),
            len(nixtla_df),
            int(nixtla_df["y"].sum()),
            ", ".join(nixtla_df.columns),
        ],
    })


def display_dataframe(title: str, dataframe: pd.DataFrame, n: int | None = None) -> None:
    """Imprime un título y despliega un dataframe."""
    print(f"\n{'=' * 90}\n{title}\n{'=' * 90}")
    display(dataframe if n is None else dataframe.head(n))


---
## 📦 PARTE 2: Carga de Datos y Formato Nixtla

El archivo CSV contiene registros individuales de dengue. Para modelar una serie temporal, estos registros se agregan por semana epidemiológica. La salida se organiza con la convención de Nixtla:

- `unique_id`: identificador de la serie.
- `ds`: fecha temporal de cada observación, en este caso el lunes de la semana epidemiológica.
- `y`: número de casos observados en esa semana.

Esta decisión es importante porque permite usar de forma consistente herramientas de pronóstico como `StatsForecast` y evita mezclar registros individuales con observaciones temporales agregadas.


In [ ]:
# Rutas candidatas para Colab y ejecución local.
DATA_PATH = resolve_existing_path([
    "/content/datos_dengue.csv",
    "datos_dengue.csv",
    "./datos_dengue.csv",
])

raw_dengue = load_raw_dengue_csv(DATA_PATH)
df = build_weekly_nixtla_series(
    raw_data=raw_dengue,
    date_column=DATE_COLUMN,
    series_id=SERIES_ID,
)

df_full = complete_weekly_calendar(df, frequency=WEEKLY_FREQUENCY)
df_full = add_modeling_target_with_seasonal_imputation(df_full)

structure_summary = summarize_nixtla_structure(df, raw_dengue)

display_dataframe("Vista inicial del CSV original", raw_dengue.head())
display_dataframe("Serie semanal en formato Nixtla", df.head())
display_dataframe("Resumen de estructura temporal", structure_summary)


In [ ]:
def plot_weekly_time_series(
    nixtla_df: pd.DataFrame,
    title: str = "Serie semanal de casos de dengue",
) -> go.Figure:
    """Grafica la serie semanal en formato Nixtla."""
    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=nixtla_df["ds"],
            y=nixtla_df["y"],
            mode="lines",
            name="Casos semanales",
            line=dict(width=1.8),
        )
    )
    fig.update_layout(
        title=title,
        xaxis_title="Fecha",
        yaxis_title="Casos semanales",
        template="plotly_white",
        height=460,
    )
    return fig

fig_series = plot_weekly_time_series(df)
fig_series.show()


### 🔍 Lectura inicial de la serie

La gráfica de la serie semanal permite identificar visualmente períodos de baja transmisión, incrementos sostenidos y brotes epidémicos. En este tipo de datos, la inspección visual no es un paso decorativo: es una herramienta de diagnóstico. Antes de aplicar modelos, conviene reconocer si la serie presenta picos extremos, cambios de nivel, persistencia temporal o posibles ciclos anuales.

El análisis posterior cuantifica estas observaciones mediante estadísticas descriptivas, detección de atípicos, medias móviles, descomposición, ACF/PACF y pruebas formales.


---
## 🔍 PARTE 3: Calidad de Datos

La calidad de datos es crítica en series temporales porque los errores no solo afectan observaciones individuales, sino también la estructura de dependencia temporal. Un hueco en el calendario, una fecha mal parseada o un pico tratado incorrectamente puede alterar la descomposición, la autocorrelación, las métricas y la comparación entre modelos.

En esta sección se revisan cuatro dimensiones:

| Dimensión | Pregunta guía | Tratamiento |
|---|---|---|
| Continuidad temporal | ¿Faltan semanas en el calendario? | Completar calendario y marcar faltantes |
| Valores nulos | ¿Hay semanas sin conteo observado? | Imputación auxiliar solo para análisis que exigen regularidad |
| Outliers | ¿Hay semanas extremadamente altas? | Detectar, reportar y conservar como posible señal epidemiológica |
| Cambios de régimen | ¿Cambian media o variabilidad por período? | Medias y desviaciones móviles; comparación por segmentos |


### 3.1 Valores Faltantes — Detección y Estrategias de Imputación

Se distingue entre una semana observada con cero casos y una semana ausente en el calendario. En vigilancia epidemiológica, esa diferencia es sustantiva: cero casos es información; una semana faltante puede reflejar ausencia de reporte, corte de datos o problema de consolidación.


In [ ]:
def summarize_calendar_quality(completed_df: pd.DataFrame, observed_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Resume continuidad semanal y devuelve las semanas faltantes."""
    missing_weeks = completed_df.loc[
        completed_df["is_missing_calendar"],
        ["unique_id", "ds", "y"],
    ].copy()
    quality_summary = pd.DataFrame({
        "Indicador": [
            "Semanas esperadas en calendario continuo",
            "Semanas observadas en CSV agregado",
            "Semanas faltantes",
            "Porcentaje de semanas faltantes",
            "Valores nulos en y observada",
        ],
        "Valor": [
            len(completed_df),
            len(observed_df),
            int(completed_df["is_missing_calendar"].sum()),
            round(completed_df["is_missing_calendar"].mean() * 100, 3),
            int(observed_df["y"].isna().sum()),
        ],
    })
    return quality_summary, missing_weeks

quality_summary, missing_weeks = summarize_calendar_quality(df_full, df)

display_dataframe("Resumen de calidad temporal", quality_summary)
display_dataframe("Semanas faltantes detectadas", missing_weeks)


### Imputación de valores faltantes por media estacional

La serie observada se conserva intacta en la columna `y`. Para análisis que requieren frecuencia completamente regular, se crea `y_model`, una variable auxiliar imputada de forma conservadora. La estrategia combina media histórica de la misma semana del año e interpolación lineal únicamente cuando es necesario.

Esta decisión evita dos errores frecuentes: eliminar semanas del calendario o reemplazar permanentemente la serie observada. En el informe final conviene reportar cuántas semanas fueron imputadas y aclarar que la imputación se usa solo para procedimientos técnicos que requieren regularidad temporal.


In [ ]:
imputation_audit = df_full.loc[
    df_full["is_missing_calendar"],
    ["unique_id", "ds", "week_of_year", "y", "y_model"],
].copy()

display_dataframe("Auditoría de imputación auxiliar para semanas faltantes", imputation_audit)


### 3.2 Detección de Outliers — IQR y Z-score

La detección de outliers se realiza con dos criterios complementarios. El método IQR identifica valores extremos respecto a la distribución global; el Z-score mide desviaciones en unidades de desviación estándar. En datos epidemiológicos, un valor atípico no equivale automáticamente a error: puede ser un brote. Por eso, los outliers se reportan y se visualizan, pero no se eliminan por defecto.


In [ ]:
def compute_descriptive_statistics(nixtla_df: pd.DataFrame) -> pd.DataFrame:
    """Calcula estadísticas descriptivas de la variable objetivo."""
    y = nixtla_df["y"].astype(float)
    return pd.DataFrame({
        "Indicador": [
            "Media", "Mediana", "Varianza", "Desviación estándar", "Coeficiente de variación",
            "Mínimo", "Máximo", "Asimetría", "Curtosis"
        ],
        "Valor": [
            y.mean(), y.median(), y.var(ddof=1), y.std(ddof=1), y.std(ddof=1) / y.mean(),
            y.min(), y.max(), skew(y, bias=False), kurtosis(y, fisher=True, bias=False)
        ],
    })


def detect_iqr_outliers(
    nixtla_df: pd.DataFrame,
    target_column: str = "y",
    iqr_multiplier: float = 1.5,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Detecta outliers por rango intercuartílico y devuelve resumen, datos marcados y top outliers."""
    result = nixtla_df.copy()
    q1 = result[target_column].quantile(0.25)
    q3 = result[target_column].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - iqr_multiplier * iqr
    upper_bound = q3 + iqr_multiplier * iqr
    result["is_outlier_iqr"] = (result[target_column] < lower_bound) | (result[target_column] > upper_bound)

    summary = pd.DataFrame({
        "Indicador": ["Q1", "Q3", "IQR", "Límite inferior", "Límite superior", "Outliers IQR", "% Outliers IQR"],
        "Valor": [q1, q3, iqr, lower_bound, upper_bound, int(result["is_outlier_iqr"].sum()), round(result["is_outlier_iqr"].mean() * 100, 3)],
    })
    top_outliers = result.loc[result["is_outlier_iqr"], ["unique_id", "ds", target_column]].sort_values(target_column, ascending=False).head(15)
    return summary, result, top_outliers


def detect_zscore_outliers(
    nixtla_df: pd.DataFrame,
    target_column: str = "y",
    threshold: float = 3.0,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Detecta outliers mediante z-score absoluto."""
    result = nixtla_df.copy()
    result["z_score"] = stats.zscore(result[target_column].astype(float), nan_policy="omit")
    result["is_outlier_zscore"] = result["z_score"].abs() > threshold
    summary = pd.DataFrame({
        "Indicador": ["Umbral |z|", "Outliers Z-score", "% Outliers Z-score"],
        "Valor": [threshold, int(result["is_outlier_zscore"].sum()), round(result["is_outlier_zscore"].mean() * 100, 3)],
    })
    return summary, result


descriptive_statistics = compute_descriptive_statistics(df)
iqr_summary, df_outliers, top_iqr_outliers = detect_iqr_outliers(df)
zscore_summary, df_zscore = detect_zscore_outliers(df)

display_dataframe("Estadísticas descriptivas de casos semanales", descriptive_statistics)
display_dataframe("Resumen de outliers por IQR", iqr_summary)
display_dataframe("Resumen de outliers por Z-score", zscore_summary)
display_dataframe("Principales semanas atípicas por IQR", top_iqr_outliers)


In [ ]:
def plot_series_with_outliers(marked_df: pd.DataFrame) -> go.Figure:
    """Grafica la serie semanal resaltando semanas atípicas por IQR."""
    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=marked_df["ds"],
            y=marked_df["y"],
            mode="lines",
            name="Serie semanal observada",
            line=dict(color="rgba(80, 120, 180, 0.65)", width=1.6),
        )
    )
    outliers = marked_df[marked_df["is_outlier_iqr"]]
    fig.add_trace(
        go.Scatter(
            x=outliers["ds"],
            y=outliers["y"],
            mode="markers",
            name="Outliers IQR",
            marker=dict(size=7, color="red", opacity=0.85),
        )
    )
    fig.update_layout(
        title="Serie semanal de dengue con semanas atípicas por criterio IQR",
        xaxis_title="Fecha",
        yaxis_title="Casos semanales",
        template="plotly_white",
        height=460,
    )
    return fig

fig_outliers = plot_series_with_outliers(df_outliers)
fig_outliers.show()


### 🔍 Interpretación — Outliers

Los outliers deben interpretarse con criterio epidemiológico. En una serie de dengue, las semanas con valores muy superiores al rango intercuartílico suelen coincidir con episodios de transmisión intensificada. Por tanto, no se recomienda eliminarlos automáticamente. En lugar de tratarlos como ruido, se deben documentar, analizar temporalmente y considerar en modelos futuros mediante transformaciones, variables exógenas, modelos robustos o enfoques específicos para brotes.

Una decisión razonable para esta fase es conservar los valores extremos en la serie observada y usar transformaciones como `log(y+1)` para reducir su influencia en pruebas y modelos sensibles a la escala.


In [ ]:
def detect_monthly_iqr_outliers(nixtla_df: pd.DataFrame, target_column: str = "y") -> pd.DataFrame:
    """Detecta outliers por IQR dentro de cada mes calendario."""
    result = nixtla_df.copy()
    result["month"] = result["ds"].dt.month
    result["is_outlier_monthly_iqr"] = False

    for month in sorted(result["month"].unique()):
        monthly_mask = result["month"] == month
        monthly_values = result.loc[monthly_mask, target_column]
        q1 = monthly_values.quantile(0.25)
        q3 = monthly_values.quantile(0.75)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        result.loc[monthly_mask, "is_outlier_monthly_iqr"] = (monthly_values < lower) | (monthly_values > upper)

    return result

monthly_outlier_df = detect_monthly_iqr_outliers(df)
monthly_outlier_summary = (
    monthly_outlier_df.groupby("month")
    .agg(
        observations=("y", "count"),
        monthly_iqr_outliers=("is_outlier_monthly_iqr", "sum"),
        max_cases=("y", "max"),
        median_cases=("y", "median"),
    )
    .reset_index()
)

display_dataframe("Resumen de outliers por mes calendario", monthly_outlier_summary)


### 3.3 Cambios de Régimen y Anomalías Estructurales

Además de los outliers individuales, interesa saber si la serie cambia de régimen: períodos prolongados con medias o varianzas distintas. Para ello se usan medias móviles y desviaciones móviles. Estas herramientas no prueban causalidad, pero permiten ubicar visualmente fases de baja transmisión, transición y brote.


In [ ]:
def add_rolling_features(
    nixtla_df: pd.DataFrame,
    windows: Sequence[int] = (4, 13, 52),
) -> pd.DataFrame:
    """Agrega medias y desviaciones móviles para distintas ventanas."""
    result = nixtla_df.copy()
    for window in windows:
        result[f"rolling_mean_{window}"] = result["y"].rolling(window=window, min_periods=1).mean()
        result[f"rolling_std_{window}"] = result["y"].rolling(window=window, min_periods=max(2, window // 3)).std()
    return result


def plot_rolling_regime_diagnostics(rolling_df: pd.DataFrame) -> go.Figure:
    """Grafica serie, medias móviles y desviación móvil anual."""
    fig = make_subplots(
        rows=2,
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.12,
        subplot_titles=(
            "Serie observada y medias móviles",
            "Desviación estándar móvil de 52 semanas",
        ),
    )

    fig.add_trace(go.Scatter(x=rolling_df["ds"], y=rolling_df["y"], mode="lines", name="Observada", line=dict(width=1, color="lightgray")), row=1, col=1)
    fig.add_trace(go.Scatter(x=rolling_df["ds"], y=rolling_df["rolling_mean_4"], mode="lines", name="Media móvil 4 semanas"), row=1, col=1)
    fig.add_trace(go.Scatter(x=rolling_df["ds"], y=rolling_df["rolling_mean_13"], mode="lines", name="Media móvil 13 semanas"), row=1, col=1)
    fig.add_trace(go.Scatter(x=rolling_df["ds"], y=rolling_df["rolling_mean_52"], mode="lines", name="Media móvil 52 semanas"), row=1, col=1)
    fig.add_trace(go.Scatter(x=rolling_df["ds"], y=rolling_df["rolling_std_52"], mode="lines", name="STD móvil 52 semanas"), row=2, col=1)

    fig.update_layout(
        title="Diagnóstico visual de tendencia, suavizamiento y cambios de régimen",
        template="plotly_white",
        height=650,
        legend=dict(orientation="h", yanchor="bottom", y=1.03),
    )
    fig.update_yaxes(title_text="Casos", row=1, col=1)
    fig.update_yaxes(title_text="Desviación", row=2, col=1)
    fig.update_xaxes(title_text="Fecha", row=2, col=1)
    return fig

rolling_df = add_rolling_features(df)
fig_regime = plot_rolling_regime_diagnostics(rolling_df)
fig_regime.show()


## Interpretación

Las medias móviles permiten separar ruido semanal de movimientos de mediano plazo. La media de 4 semanas conserva variaciones rápidas, la de 13 semanas resume aproximadamente un trimestre epidemiológico y la de 52 semanas aproxima el nivel anual. Cuando estas curvas cambian de nivel de manera sostenida, hay evidencia visual de cambios de régimen.

En dengue, estos cambios pueden relacionarse con ciclos epidémicos, condiciones climáticas, acumulación de susceptibles, variación en vigilancia o cambios en intervención sanitaria. Para la fase actual, el objetivo es diagnosticar esta estructura; en fases posteriores, puede traducirse en modelos con diferenciación, estacionalidad, variables exógenas o validación por ventanas.


In [ ]:
def compare_temporal_segments(
    nixtla_df: pd.DataFrame,
    split_date: str | pd.Timestamp,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Compara media y varianza antes/después de una fecha de corte."""
    split_timestamp = pd.to_datetime(split_date)
    segmented = nixtla_df.copy()
    segmented["segment"] = np.where(segmented["ds"] < split_timestamp, "Antes del corte", "Después del corte")

    segment_summary = (
        segmented.groupby("segment")
        .agg(
            observations=("y", "count"),
            mean_cases=("y", "mean"),
            median_cases=("y", "median"),
            variance_cases=("y", "var"),
            max_cases=("y", "max"),
        )
        .reset_index()
    )

    before = segmented.loc[segmented["segment"] == "Antes del corte", "y"]
    after = segmented.loc[segmented["segment"] == "Después del corte", "y"]

    tests = pd.DataFrame({
        "Prueba": ["Levene varianzas", "t-test Welch medias"],
        "Estadístico": [
            stats.levene(before, after, center="median").statistic,
            stats.ttest_ind(before, after, equal_var=False).statistic,
        ],
        "p-value": [
            stats.levene(before, after, center="median").pvalue,
            stats.ttest_ind(before, after, equal_var=False).pvalue,
        ],
    })

    return segment_summary, tests

# Corte sugerido por defecto: inicio del último 20% temporal, consistente con train-test split.
default_split_position = int(np.floor(len(df) * (1 - DEFAULT_TEST_SIZE_RATIO)))
default_regime_split_date = df.iloc[default_split_position]["ds"]

segment_summary, segment_tests = compare_temporal_segments(df, default_regime_split_date)

display_dataframe(f"Resumen por segmentos usando corte {default_regime_split_date.date()}", segment_summary)
display_dataframe("Pruebas simples de cambio de media/varianza", segment_tests)


### 🔍 Interpretación — Cambios de Régimen

Las pruebas de comparación por segmentos son diagnósticas, no definitivas. Una diferencia significativa de medias o varianzas sugiere que entrenar un único modelo global puede ocultar comportamientos distintos entre períodos. Sin embargo, en epidemiología los cambios de régimen no siempre son permanentes: algunos brotes son transitorios y otros pueden iniciar ciclos prolongados.

Por esta razón, el tratamiento recomendado no es eliminar periodos extremos, sino evaluar modelos con memoria temporal, estacionalidad y validación cruzada temporal. La segmentación puede usarse como análisis complementario para entender cuándo un modelo baseline falla y qué tipo de modelo avanzado podría mejorar.


---
## 📈 PARTE 4: Análisis de Autocorrelación (ACF)

La autocorrelación mide cuánto se parece la serie a sí misma después de distintos rezagos. En una serie de enfermedades infecciosas, una autocorrelación alta en rezagos cortos es esperable: una semana de alta incidencia suele estar seguida por semanas de incidencia elevada.

Se complementa la ACF con la PACF para distinguir dependencia directa de dependencia mediada por rezagos intermedios. Esta lectura será útil en fases posteriores para orientar modelos ARIMA/SARIMA.


In [ ]:
def plot_acf_pacf_plotly(
    series: pd.Series,
    max_lag: int = 60,
    title: str = "ACF y PACF de la serie semanal",
) -> go.Figure:
    """Grafica ACF y PACF con bandas aproximadas de confianza."""
    clean_series = pd.Series(series).dropna().astype(float)
    acf_values = acf(clean_series, nlags=max_lag, fft=True)
    pacf_values = pacf(clean_series, nlags=max_lag, method="ywm")
    lags = np.arange(max_lag + 1)
    confidence_limit = 1.96 / np.sqrt(len(clean_series))

    fig = make_subplots(rows=1, cols=2, subplot_titles=("ACF", "PACF"))

    for col, values, name in [(1, acf_values, "ACF"), (2, pacf_values, "PACF")]:
        fig.add_trace(go.Bar(x=lags, y=values, name=name, showlegend=False), row=1, col=col)
        fig.add_hline(y=confidence_limit, line_dash="dash", line_color="red", row=1, col=col)
        fig.add_hline(y=-confidence_limit, line_dash="dash", line_color="red", row=1, col=col)
        fig.update_xaxes(title_text="Rezago", row=1, col=col)
        fig.update_yaxes(title_text="Correlación", row=1, col=col)

    fig.update_layout(title=title, template="plotly_white", height=440)
    return fig

fig_acf_pacf = plot_acf_pacf_plotly(df_full["y_model"], max_lag=60)
fig_acf_pacf.show()


In [ ]:
def compute_acf_pacf_significance_table(series: pd.Series, max_lag: int = 60) -> pd.DataFrame:
    """Calcula ACF/PACF y marca rezagos significativos con regla ±1.96/sqrt(n)."""
    clean_series = pd.Series(series).dropna().astype(float)
    confidence_limit = 1.96 / np.sqrt(len(clean_series))
    acf_values = acf(clean_series, nlags=max_lag, fft=True)
    pacf_values = pacf(clean_series, nlags=max_lag, method="ywm")
    table = pd.DataFrame({
        "lag": np.arange(max_lag + 1),
        "ACF": acf_values,
        "PACF": pacf_values,
    })
    table["ACF_significativa"] = table["ACF"].abs() > confidence_limit
    table["PACF_significativa"] = table["PACF"].abs() > confidence_limit
    table["limite_confianza_aprox"] = confidence_limit
    return table

acf_pacf_table = compute_acf_pacf_significance_table(df_full["y_model"], max_lag=60)

display_dataframe("Primeros rezagos ACF/PACF", acf_pacf_table.head(30))
display_dataframe("Rezagos significativos ACF", acf_pacf_table.query("lag > 0 and ACF_significativa").head(25))
display_dataframe("Rezagos significativos PACF", acf_pacf_table.query("lag > 0 and PACF_significativa").head(25))


## Interpretación — Autocorrelación

Una ACF con valores significativos en rezagos bajos indica persistencia temporal: los casos recientes contienen información sobre los casos futuros. Si el decaimiento es lento, puede existir no estacionariedad o ciclos prolongados. Si aparecen rezagos significativos cerca de 52 semanas, hay indicios de patrón anual.

La PACF permite identificar rezagos con asociación parcial relevante, una guía inicial para órdenes autorregresivos. No se debe seleccionar un modelo ARIMA solo por inspección visual, pero esta lectura ayuda a formular hipótesis para la siguiente fase.


---
## 📐 PARTE 4.5: Tests de Estacionaridad — ADF y KPSS

La estacionariedad implica que propiedades como media, varianza y autocorrelación se mantienen relativamente estables en el tiempo. Muchos modelos clásicos, especialmente ARIMA, requieren que la serie sea estacionaria o se transforme hasta aproximarla.

Se evalúan varias transformaciones: serie original, `log(y+1)`, diferencia simple, diferencia estacional de 52 semanas, log con diferencia simple y raíz cuadrada. ADF y KPSS se interpretan de forma complementaria:

- ADF: hipótesis nula de raíz unitaria/no estacionariedad.
- KPSS: hipótesis nula de estacionariedad.


In [ ]:
def build_stationarity_dataset(completed_df: pd.DataFrame, model_column: str = "y_model") -> pd.DataFrame:
    """Construye transformaciones comunes para diagnóstico de estacionariedad."""
    result = completed_df[["unique_id", "ds", model_column]].rename(columns={model_column: "y"}).copy()
    result["log_y"] = np.log1p(result["y"])
    result["sqrt_y"] = np.sqrt(result["y"])
    result["diff_y"] = result["y"].diff()
    result["seasonal_diff_y"] = result["y"].diff(SEASON_LENGTH)
    result["log_diff_y"] = result["log_y"].diff()
    return result


def run_adf_kpss_tests(series: pd.Series, transformation_name: str) -> dict:
    """Ejecuta ADF y KPSS para una transformación específica."""
    clean_series = pd.Series(series).dropna().astype(float)
    adf_result = adfuller(clean_series, autolag="AIC")

    try:
        kpss_result = kpss(clean_series, regression="c", nlags="auto")
        kpss_statistic = kpss_result[0]
        kpss_p_value = kpss_result[1]
    except Exception:
        kpss_statistic = np.nan
        kpss_p_value = np.nan

    mean_value = clean_series.mean()
    std_value = clean_series.std(ddof=1)
    coefficient_variation = std_value / mean_value if mean_value != 0 else np.nan

    return {
        "Transformación": transformation_name,
        "n": len(clean_series),
        "Media": mean_value,
        "Desv. estándar": std_value,
        "CV": coefficient_variation,
        "ADF estadístico": adf_result[0],
        "ADF p-value": adf_result[1],
        "KPSS estadístico": kpss_statistic,
        "KPSS p-value": kpss_p_value,
    }

stationarity_df = build_stationarity_dataset(df_full)

stationarity_summary = pd.DataFrame([
    run_adf_kpss_tests(stationarity_df["y"], "Original"),
    run_adf_kpss_tests(stationarity_df["log_y"], "Log(y+1)"),
    run_adf_kpss_tests(stationarity_df["sqrt_y"], "Raíz cuadrada"),
    run_adf_kpss_tests(stationarity_df["diff_y"], "Diferencia simple"),
    run_adf_kpss_tests(stationarity_df["seasonal_diff_y"], "Diferencia estacional 52"),
    run_adf_kpss_tests(stationarity_df["log_diff_y"], "Log(y+1) + diferencia simple"),
])

display_dataframe("Resumen ADF/KPSS por transformación", stationarity_summary)


In [ ]:
def plot_stationarity_transformations(stationarity_data: pd.DataFrame) -> go.Figure:
    """Visualiza transformaciones clave para evaluar estacionariedad."""
    fig = make_subplots(
        rows=3,
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.10,
        subplot_titles=("Serie original", "Transformación log(y+1)", "Log(y+1) con diferencia simple"),
    )
    fig.add_trace(go.Scatter(x=stationarity_data["ds"], y=stationarity_data["y"], mode="lines", name="Original"), row=1, col=1)
    fig.add_trace(go.Scatter(x=stationarity_data["ds"], y=stationarity_data["log_y"], mode="lines", name="Log(y+1)"), row=2, col=1)
    fig.add_trace(go.Scatter(x=stationarity_data["ds"], y=stationarity_data["log_diff_y"], mode="lines", name="Log diff"), row=3, col=1)
    fig.update_layout(title="Comparación visual de transformaciones", template="plotly_white", height=720, showlegend=False)
    fig.update_xaxes(title_text="Fecha", row=3, col=1)
    fig.update_yaxes(title_text="Casos", row=1, col=1)
    fig.update_yaxes(title_text="Log", row=2, col=1)
    fig.update_yaxes(title_text="Diferencia", row=3, col=1)
    return fig

fig_transformations = plot_stationarity_transformations(stationarity_df)
fig_transformations.show()


### 🔍 Interpretación — Tests de Estacionaridad

La decisión no debe basarse en un único p-valor. En series reales, ADF y KPSS pueden ofrecer señales parcialmente contradictorias, especialmente cuando existen brotes, cambios de régimen o estacionalidad. La lectura recomendada combina evidencia visual, ACF/PACF, estadísticos y criterio de interpretabilidad.

Para las siguientes fases, `log(y+1)` suele ser una transformación útil porque estabiliza la escala y reduce el dominio de picos extremos sin perder interpretabilidad. La diferenciación simple o estacional debe evaluarse según el modelo específico, especialmente en ARIMA/SARIMA.


---
## 🔬 PARTE 5: Test de Ljung-Box

El test de Ljung-Box evalúa si un conjunto de autocorrelaciones es simultáneamente igual a cero. En la serie original, permite verificar si existe estructura temporal aprovechable. En residuos de modelos, permite evaluar si el modelo dejó patrones sin capturar.

La hipótesis nula es que no hay autocorrelación conjunta hasta el rezago evaluado. Valores p bajos sugieren que la serie no se comporta como ruido blanco.


In [ ]:
def run_ljung_box(
    series: pd.Series,
    lags: Sequence[int],
    label: str,
) -> pd.DataFrame:
    """Ejecuta Ljung-Box para un conjunto de rezagos y retorna tabla ordenada."""
    result = acorr_ljungbox(pd.Series(series).dropna().astype(float), lags=list(lags), return_df=True)
    result = result.reset_index().rename(columns={"index": "Lag", "lb_stat": "Estadístico Q", "lb_pvalue": "p-value"})
    result.insert(0, "Serie", label)
    return result

ljung_lags = [1, 4, 8, 12, 24, 52]
ljung_original = run_ljung_box(stationarity_df["y"], ljung_lags, "Original")
ljung_log_diff = run_ljung_box(stationarity_df["log_diff_y"], ljung_lags[:-1], "Log(y+1) + diferencia")
ljung_results = pd.concat([ljung_original, ljung_log_diff], ignore_index=True)

display_dataframe("Test de Ljung-Box en serie original y transformación recomendada", ljung_results)


In [ ]:
def plot_ljung_box_pvalues(ljung_df: pd.DataFrame) -> go.Figure:
    """Grafica p-values de Ljung-Box por rezago y serie/transformación."""
    fig = go.Figure()
    for label in ljung_df["Serie"].unique():
        data = ljung_df[ljung_df["Serie"] == label]
        fig.add_trace(go.Scatter(x=data["Lag"], y=data["p-value"], mode="lines+markers", name=label))
    fig.add_hline(y=0.05, line_dash="dash", line_color="red", annotation_text="α = 0.05")
    fig.update_layout(
        title="p-values del test de Ljung-Box por rezago",
        xaxis_title="Rezago",
        yaxis_title="p-value",
        template="plotly_white",
        height=430,
    )
    return fig

fig_ljung = plot_ljung_box_pvalues(ljung_results)
fig_ljung.show()


### 🔍 Interpretación — Ljung-Box

Si los p-valores son menores a 0.05, se rechaza la hipótesis de ruido blanco. Esto confirma que la serie conserva dependencia temporal y que los modelos de pronóstico tienen información útil para explotar. Cuando la transformación reduce la autocorrelación, se obtiene evidencia de que el preprocesamiento mejora la estacionariedad, aunque no necesariamente elimina toda la estructura temporal.


---
## ✂️ PARTE 6: Train-Test Split Temporal

En series temporales no se debe aleatorizar la partición. El entrenamiento debe ocurrir antes que la prueba para simular un escenario real de pronóstico: se aprende con el pasado y se evalúa contra el futuro.

Se usa la columna `y_model` únicamente para garantizar continuidad semanal. La columna original `y` sigue siendo la referencia de calidad e interpretación.


In [ ]:
def create_temporal_train_test_split(
    completed_df: pd.DataFrame,
    model_column: str = "y_model",
    test_size_ratio: float = DEFAULT_TEST_SIZE_RATIO,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Crea train/test temporal y devuelve resumen."""
    model_df = completed_df[["unique_id", "ds", model_column]].rename(columns={model_column: "y"}).copy()
    model_df = model_df.sort_values("ds").reset_index(drop=True)

    test_size = int(np.ceil(len(model_df) * test_size_ratio))
    train_df = model_df.iloc[:-test_size].copy()
    test_df = model_df.iloc[-test_size:].copy()

    split_summary = pd.DataFrame({
        "Segmento": ["Entrenamiento", "Prueba"],
        "Observaciones": [len(train_df), len(test_df)],
        "Inicio": [train_df["ds"].min().date(), test_df["ds"].min().date()],
        "Fin": [train_df["ds"].max().date(), test_df["ds"].max().date()],
    })
    return model_df, train_df, test_df, split_summary

model_df, train, test, split_summary = create_temporal_train_test_split(df_full)
fecha_corte = test["ds"].min()

display_dataframe("Resumen de partición temporal", split_summary)


---
## 🤖 PARTE 7: Baselines con `statsforecast`

Los modelos baseline establecen el nivel mínimo de desempeño que debe superar cualquier modelo avanzado. Se implementan cuatro referencias:

| Modelo | Idea | Utilidad |
|---|---|---|
| `Naive` | Repite el último valor observado | Referencia mínima de persistencia |
| `SeasonalNaive` | Usa el valor del mismo rezago estacional | Captura patrón anual simple |
| `WindowAverage` | Promedia una ventana reciente | Suaviza ruido de corto plazo |
| `RandomWalkWithDrift` | Proyecta tendencia lineal simple | Captura desplazamiento promedio |

Se conserva el flujo estándar de Nixtla: `fit(train)` → `predict(h)`.


In [ ]:
def fit_predict_baselines_with_statsforecast(
    train_df: pd.DataFrame,
    horizon: int,
    frequency: str = WEEKLY_FREQUENCY,
    season_length: int = SEASON_LENGTH,
    window_size: int = 4,
) -> pd.DataFrame:
    """Ajusta modelos baseline con StatsForecast y pronostica h pasos."""
    models = [
        Naive(),
        SeasonalNaive(season_length=season_length),
        WindowAverage(window_size=window_size),
        RandomWalkWithDrift(),
    ]
    forecaster = StatsForecast(models=models, freq=frequency, n_jobs=-1)
    forecaster.fit(train_df)
    predictions = forecaster.predict(h=horizon)
    return predictions

preds = fit_predict_baselines_with_statsforecast(train, horizon=len(test), window_size=4)
test_preds = test.merge(preds, on=["unique_id", "ds"], how="left")

display_dataframe("Predicciones baseline - primeras filas", test_preds.head())
display_dataframe("Predicciones baseline - últimas filas", test_preds.tail())


In [ ]:
def plot_baseline_forecasts(
    historical_df: pd.DataFrame,
    test_predictions: pd.DataFrame,
    cutoff_date: pd.Timestamp,
) -> go.Figure:
    """Compara valores reales y predicciones baseline sobre el conjunto de prueba."""
    model_columns = ["Naive", "SeasonalNaive", "WindowAverage", "RWD"]
    model_labels = {
        "Naive": "Naive",
        "SeasonalNaive": "Seasonal Naive (52)",
        "WindowAverage": "Media móvil (4)",
        "RWD": "Random Walk with Drift",
    }

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=historical_df["ds"], y=historical_df["y"], mode="lines", name="Histórico",
        line=dict(color="rgba(160,160,160,0.55)", width=1.4),
    ))
    fig.add_trace(go.Scatter(
        x=test_predictions["ds"], y=test_predictions["y"], mode="lines+markers", name="Real (test)",
        line=dict(color="black", width=2.4), marker=dict(size=5),
    ))
    for model_column in model_columns:
        fig.add_trace(go.Scatter(
            x=test_predictions["ds"], y=test_predictions[model_column], mode="lines",
            name=model_labels[model_column], line=dict(width=2, dash="dot"),
        ))
    fig.add_shape(
        type="line", xref="x", yref="paper",
        x0=cutoff_date, x1=cutoff_date, y0=0, y1=1,
        line=dict(dash="dash", color="gray", width=1.5),
    )
    fig.add_annotation(
        x=cutoff_date, y=1.04, yref="paper", text="Inicio test",
        showarrow=False, font=dict(size=10, color="gray"), xanchor="left",
    )
    fig.update_layout(
        title="Comparación de modelos baseline en el conjunto de prueba",
        xaxis_title="Fecha", yaxis_title="Casos semanales",
        template="plotly_white", height=500,
        legend=dict(orientation="h", yanchor="bottom", y=1.05),
    )
    return fig

fig_baselines = plot_baseline_forecasts(model_df, test_preds, fecha_corte)
fig_baselines.show()


### 🔍 ¿Qué observar en los pronósticos?

El gráfico permite evaluar si los modelos capturan nivel, tendencia y brotes. Un baseline puede ser competitivo en periodos estables, pero fallar ante cambios abruptos. Este comportamiento no invalida el baseline; por el contrario, ayuda a diagnosticar qué estructuras quedan pendientes para modelos avanzados.

Si `SeasonalNaive` funciona relativamente bien, la estacionalidad anual tiene peso. Si `WindowAverage` domina, la información reciente es más útil que el patrón anual. Si `RWD` falla, una tendencia lineal global no resume adecuadamente la dinámica epidemiológica.


---
## 📊 PARTE 8: Métricas de Evaluación

Las métricas cuantifican el desempeño observado en el gráfico. Se usan métricas complementarias:

| Métrica | Interpretación | Precaución |
|---|---|---|
| MAE | Error absoluto promedio en casos | Fácil de interpretar |
| RMSE | Penaliza más los errores grandes | Sensible a brotes extremos |
| MAPE | Error porcentual medio | Problemático con valores cercanos a cero |
| sMAPE | Error porcentual simétrico | Más estable que MAPE, pero no perfecto |


In [ ]:
def mean_absolute_error(y_true: Sequence[float], y_pred: Sequence[float]) -> float:
    return float(np.mean(np.abs(np.asarray(y_true) - np.asarray(y_pred))))


def root_mean_squared_error(y_true: Sequence[float], y_pred: Sequence[float]) -> float:
    errors = np.asarray(y_true) - np.asarray(y_pred)
    return float(np.sqrt(np.mean(errors ** 2)))


def mean_absolute_percentage_error(y_true: Sequence[float], y_pred: Sequence[float]) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = y_true != 0
    return float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100)


def symmetric_mean_absolute_percentage_error(y_true: Sequence[float], y_pred: Sequence[float]) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2
    mask = denominator != 0
    return float(np.mean(np.abs(y_true[mask] - y_pred[mask]) / denominator[mask]) * 100)


def evaluate_forecast_models(
    prediction_df: pd.DataFrame,
    model_columns: Sequence[str],
    target_column: str = "y",
) -> pd.DataFrame:
    """Calcula métricas por modelo y ordena por RMSE."""
    rows = []
    for model_column in model_columns:
        rows.append({
            "Modelo": model_column,
            "MAE": mean_absolute_error(prediction_df[target_column], prediction_df[model_column]),
            "RMSE": root_mean_squared_error(prediction_df[target_column], prediction_df[model_column]),
            "MAPE (%)": mean_absolute_percentage_error(prediction_df[target_column], prediction_df[model_column]),
            "sMAPE (%)": symmetric_mean_absolute_percentage_error(prediction_df[target_column], prediction_df[model_column]),
        })
    return pd.DataFrame(rows).sort_values("RMSE").reset_index(drop=True)

model_columns = ["Naive", "SeasonalNaive", "WindowAverage", "RWD"]
metrics_summary = evaluate_forecast_models(test_preds, model_columns)

display_dataframe("Métricas de desempeño en conjunto de prueba", metrics_summary)


In [ ]:
def plot_metrics_comparison(metrics_df: pd.DataFrame, metric: str = "RMSE") -> go.Figure:
    """Grafica comparación de modelos para una métrica seleccionada."""
    fig = go.Figure()
    fig.add_trace(go.Bar(x=metrics_df["Modelo"], y=metrics_df[metric], text=metrics_df[metric].round(2), textposition="outside"))
    fig.update_layout(
        title=f"Comparación de modelos baseline por {metric}",
        xaxis_title="Modelo",
        yaxis_title=metric,
        template="plotly_white",
        height=430,
    )
    return fig

fig_rmse = plot_metrics_comparison(metrics_summary, metric="RMSE")
fig_rmse.show()


### 🔍 Conclusión de métricas

La selección del mejor baseline debe hacerse con una métrica principal y una lectura complementaria. RMSE es útil cuando interesa penalizar fuertemente la subestimación de brotes; MAE es más estable ante extremos; MAPE y sMAPE ayudan a comparar en términos relativos, pero deben interpretarse con cuidado cuando hay semanas de baja incidencia.

El mejor baseline en esta fase no debe verse como modelo final, sino como umbral mínimo. Cualquier modelo ARIMA/SARIMA, suavización exponencial o modelo avanzado debería superar este desempeño para justificar su complejidad.


---
## 🔬 Cierre del Ciclo: Ljung-Box sobre Residuos del Mejor Modelo

El diagnóstico de residuos responde una pregunta clave: después de aplicar el mejor baseline, ¿queda estructura temporal sin capturar? Si los residuos todavía muestran autocorrelación, hay margen para modelos más sofisticados.


In [ ]:
def compute_model_residuals(
    prediction_df: pd.DataFrame,
    model_name: str,
    target_column: str = "y",
) -> pd.DataFrame:
    """Calcula residuos de un modelo específico."""
    residuals = prediction_df[["unique_id", "ds", target_column, model_name]].copy()
    residuals["residual"] = residuals[target_column] - residuals[model_name]
    return residuals

best_model = metrics_summary.loc[0, "Modelo"]
residuals_df = compute_model_residuals(test_preds, best_model)

residual_summary = pd.DataFrame({
    "Indicador": [
        "Mejor modelo por RMSE", "Media de residuos", "Desv. estándar de residuos", "Mínimo residuo", "Máximo residuo"
    ],
    "Valor": [
        best_model,
        residuals_df["residual"].mean(),
        residuals_df["residual"].std(ddof=1),
        residuals_df["residual"].min(),
        residuals_df["residual"].max(),
    ],
})

ljung_residuals = run_ljung_box(residuals_df["residual"], [1, 4, 8, 12, 24], f"Residuos {best_model}")

display_dataframe("Resumen de residuos del mejor baseline", residual_summary)
display_dataframe("Ljung-Box sobre residuos del mejor baseline", ljung_residuals)


In [ ]:
def plot_model_residuals(residuals: pd.DataFrame, model_name: str) -> go.Figure:
    """Grafica residuos de un modelo en el tiempo."""
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=residuals["ds"], y=residuals["residual"], mode="lines+markers", name="Residuo"))
    fig.add_hline(y=0, line_dash="dash", line_color="gray")
    fig.update_layout(
        title=f"Residuos del mejor baseline: {model_name}",
        xaxis_title="Fecha",
        yaxis_title="Residuo",
        template="plotly_white",
        height=430,
    )
    return fig

fig_residuals = plot_model_residuals(residuals_df, best_model)
fig_residuals.show()


## Interpretación — Residuos

Si los residuos presentan rachas, picos o autocorrelación significativa, el baseline no capturó toda la estructura temporal. Esto es especialmente probable en series epidémicas, donde los brotes pueden superar la capacidad de modelos simples basados en persistencia, promedio o estacionalidad fija.

Este resultado justifica avanzar hacia modelos con componentes autorregresivos, diferenciación, estacionalidad flexible o variables exógenas.


---
## 🔄 PARTE 9: Time Series Cross-Validation

Un único split temporal puede depender demasiado del periodo específico elegido como test. La validación cruzada temporal evalúa el desempeño en múltiples ventanas ordenadas, respetando la dirección del tiempo. Esto permite revisar estabilidad, sensibilidad a brotes y consistencia del ranking de modelos.


In [ ]:
def create_temporal_cv_splits(
    data: pd.DataFrame,
    initial_train_size: int,
    horizon: int,
    step: int,
) -> list[tuple[np.ndarray, np.ndarray]]:
    """Genera splits de validación cruzada temporal expansiva."""
    splits = []
    start_test = initial_train_size
    n_observations = len(data)

    while start_test + horizon <= n_observations:
        train_index = np.arange(0, start_test)
        test_index = np.arange(start_test, start_test + horizon)
        splits.append((train_index, test_index))
        start_test += step

    return splits


def run_temporal_cross_validation(
    model_data: pd.DataFrame,
    model_columns: Sequence[str],
    initial_train_ratio: float = 0.60,
    horizon: int = 52,
    step: int = 26,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Ejecuta CV temporal para los baselines y resume métricas."""
    splits = create_temporal_cv_splits(
        model_data,
        initial_train_size=int(len(model_data) * initial_train_ratio),
        horizon=horizon,
        step=step,
    )

    rows = []
    for fold_number, (train_index, test_index) in enumerate(splits, start=1):
        cv_train = model_data.iloc[train_index].copy()
        cv_test = model_data.iloc[test_index].copy()
        cv_preds = fit_predict_baselines_with_statsforecast(cv_train, horizon=len(cv_test), window_size=4)
        cv_test_preds = cv_test.merge(cv_preds, on=["unique_id", "ds"], how="left")

        for model_column in model_columns:
            rows.append({
                "fold": fold_number,
                "Modelo": model_column,
                "Inicio test": cv_test["ds"].min().date(),
                "Fin test": cv_test["ds"].max().date(),
                "MAE": mean_absolute_error(cv_test_preds["y"], cv_test_preds[model_column]),
                "RMSE": root_mean_squared_error(cv_test_preds["y"], cv_test_preds[model_column]),
                "MAPE (%)": mean_absolute_percentage_error(cv_test_preds["y"], cv_test_preds[model_column]),
                "sMAPE (%)": symmetric_mean_absolute_percentage_error(cv_test_preds["y"], cv_test_preds[model_column]),
            })

    cv_results = pd.DataFrame(rows)
    cv_summary = (
        cv_results.groupby("Modelo")
        .agg(
            MAE_promedio=("MAE", "mean"),
            RMSE_promedio=("RMSE", "mean"),
            MAPE_promedio=("MAPE (%)", "mean"),
            sMAPE_promedio=("sMAPE (%)", "mean"),
            RMSE_desv=("RMSE", "std"),
            folds=("fold", "nunique"),
        )
        .reset_index()
        .sort_values("RMSE_promedio")
    )
    return cv_results, cv_summary

cv_results_df, cv_summary = run_temporal_cross_validation(model_df, model_columns)

display_dataframe("Resultados de validación cruzada temporal", cv_results_df)
display_dataframe("Resumen promedio de validación cruzada temporal", cv_summary)


In [ ]:
def plot_cv_summary(cv_summary_df: pd.DataFrame) -> go.Figure:
    """Grafica RMSE promedio de validación cruzada temporal."""
    fig = go.Figure()
    fig.add_trace(go.Bar(
        x=cv_summary_df["Modelo"],
        y=cv_summary_df["RMSE_promedio"],
        error_y=dict(type="data", array=cv_summary_df["RMSE_desv"].fillna(0)),
        text=cv_summary_df["RMSE_promedio"].round(2),
        textposition="outside",
    ))
    fig.update_layout(
        title="RMSE promedio en validación cruzada temporal",
        xaxis_title="Modelo",
        yaxis_title="RMSE promedio",
        template="plotly_white",
        height=430,
    )
    return fig

fig_cv = plot_cv_summary(cv_summary)
fig_cv.show()


### 🔍 Interpretación — Cross-Validation

La validación cruzada temporal es más robusta que un único split porque expone los modelos a diferentes contextos históricos. Si un modelo tiene buen promedio pero alta desviación del RMSE, su desempeño es inestable. Si mantiene bajo error en varios folds, es una referencia más confiable para fases posteriores.

En series con brotes, es común que el ranking de modelos cambie según el periodo evaluado. Por eso, esta sección permite diferenciar un modelo que funciona solo en un tramo favorable de otro que generaliza mejor en distintos escenarios temporales.


---
## 🎯 PARTE 10: Práctica Guiada

Esta sección conserva el espíritu del notebook original, pero adapta los ejercicios al dataset de dengue. La práctica busca que el análisis no se limite a ejecutar funciones, sino que conecte resultados estadísticos con decisiones de modelado.

### Ejercicios sugeridos

1. **Outliers por contexto temporal:** compara outliers globales IQR contra outliers por mes calendario. ¿Cuáles parecen brotes reales y cuáles dependen de la estacionalidad?
2. **ACF estacional:** identifica si los rezagos cercanos a 52 semanas son significativos. ¿Qué implica para SARIMA?
3. **Ljung-Box sobre residuos:** revisa si el mejor baseline deja autocorrelación. ¿Qué estructura temporal quedó pendiente?
4. **Optimización de ventana:** prueba `WindowAverage` con ventanas de 2, 4, 8, 13 y 26 semanas. ¿Cuál minimiza RMSE?
5. **Reflexión epidemiológica:** ¿qué variables exógenas podrían mejorar el pronóstico de dengue? Considera clima, lluvias, temperatura, humedad, intervención vectorial o cambios en vigilancia.


In [ ]:
def evaluate_window_average_candidates(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    candidate_windows: Sequence[int] = (2, 4, 8, 13, 26),
) -> pd.DataFrame:
    """Evalúa varias ventanas para WindowAverage manteniendo el mismo split."""
    rows = []
    for window_size in candidate_windows:
        preds_window = fit_predict_baselines_with_statsforecast(
            train_df,
            horizon=len(test_df),
            window_size=window_size,
        )
        test_preds_window = test_df.merge(preds_window, on=["unique_id", "ds"], how="left")
        rows.append({
            "window_size": window_size,
            "MAE": mean_absolute_error(test_preds_window["y"], test_preds_window["WindowAverage"]),
            "RMSE": root_mean_squared_error(test_preds_window["y"], test_preds_window["WindowAverage"]),
            "MAPE (%)": mean_absolute_percentage_error(test_preds_window["y"], test_preds_window["WindowAverage"]),
            "sMAPE (%)": symmetric_mean_absolute_percentage_error(test_preds_window["y"], test_preds_window["WindowAverage"]),
        })
    return pd.DataFrame(rows).sort_values("RMSE").reset_index(drop=True)

window_optimization_results = evaluate_window_average_candidates(train, test)

display_dataframe("Optimización de ventana para WindowAverage", window_optimization_results)


---
## ✅ Resumen de la Sesión

| Componente | Herramienta | Salida esperada |
|---|---|---|
| Formato Nixtla | `pandas` | `unique_id`, `ds`, `y` |
| Calidad temporal | calendario semanal | semanas faltantes e imputación auxiliar |
| Outliers | IQR / Z-score | semanas atípicas documentadas |
| Tendencia y régimen | medias móviles | cambios de nivel y variabilidad |
| Estacionariedad | ADF / KPSS | diagnóstico por transformación |
| Autocorrelación | ACF / PACF | rezagos significativos |
| Ljung-Box | `statsmodels` | evidencia de ruido blanco o dependencia |
| Baselines | `StatsForecast` | pronósticos reproducibles |
| Métricas | funciones propias | MAE, RMSE, MAPE, sMAPE |
| Residuos | Ljung-Box residual | estructura no capturada |
| Cross-validation | ventanas temporales | estabilidad del desempeño |

### Cierre metodológico

El notebook deja lista la base para la siguiente fase del proyecto. Si los baselines no capturan brotes o dejan residuos autocorrelacionados, hay justificación técnica para probar modelos ARIMA/SARIMA, suavización exponencial o modelos avanzados con variables exógenas. El objetivo de esta fase no es obtener el modelo final, sino construir una línea base sólida, trazable y defendible.
